In [ ]:
import ROOT
import numpy as np
import pandas as pd

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    dx=10e-6
    return (model(p + dx, par) - model(p, par)) / dx

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati (2)")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy()

#definizione variabili
p = df[:, 6]
q = df[:, 7]
err_p = df[:, 8]
err_q = df[:,9]

#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0)
model_fit.SetParLimits(0, 0, 50) 
model_fit.SetParLimits(1, 0, 50)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati con errori; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerColor(ROOT.kBlue)
gr1.SetLineColor(ROOT.kBlack)

fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

allah = [model_fit.Eval(i) for i in p]
print(allah)

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("APX")

c1.Draw()
c1.SaveAs("fit_root.png")

[  70.8   71.8   72.8   73.8   74.8   75.8   76.8   77.8   78.8   79.8
   80.8   81.8   82.8   83.8   84.8   85.8   86.8   87.8   88.8   89.8
   90.8   91.8   92.8   93.8   94.8   95.8   96.8   97.8   98.8   99.8
  100.8  101.8  102.8  103.8  104.8  105.8  106.8  107.8  108.8  109.8
  110.8  111.8  112.8  113.8  114.8  115.8  117.8  119.8  121.8  123.8
  125.8  127.8  129.8  131.8  133.8  135.8  137.8  139.8  141.8  143.8
  145.8  147.8  149.8  151.8  154.8  157.8  160.8  163.8  166.8  169.8
  172.8  175.8  179.8  183.8  187.8  191.8  195.8  199.8  203.8  207.8
  212.8  217.8  222.8  227.8  232.8  237.8  242.8  247.8  254.8  261.8
  268.8  275.8  285.8  295.8  305.8  315.8  325.8  335.8  345.8  355.8
  365.8  375.8  395.8  415.8  435.8  455.8  475.8  495.8  515.8  535.8
  555.8  575.8  625.8  675.8  725.8  775.8  825.8  875.8  975.8 1075.8
 1175.8 1275.8 1375.8 1475.8]
[1513.  1512.  1448.  1428.  1351.  1316.  1255.5 1207.  1190.  1144.5
 1109.5 1088.  1059.5 1035.5 1010.5  985.   964

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1


In [224]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if par[0] < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if par[1] < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A è compatibile con 0
---
B NON è compatibile con 0


In [225]:
residuals_norm = (q - model(p, par)) / err_qeff


print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kGreen+2)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kBlue)
hist.SetFillColor(ROOT.kBlue)
hist.GetXaxis().SetTitle("valore")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "SR")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

c2.Draw()
c2.SaveAs("hist.png")

[954.04483789 953.51077587 919.3308062  908.64956568 867.52678967
 848.83461876 816.52386618 790.62185791 781.54280347 757.24298128
 738.55081037 727.06847681 711.84770907 699.03022044 685.67866979
 672.06008813 661.11181659 649.095421   641.88558365 626.13075388
 622.12528869 608.50670702 599.69468359 587.94531902 582.87172977
 576.46298546 565.78174494 557.77081455 549.49285314 541.74895376
 531.33474425 528.66443412 522.52272082 516.11397651 510.50632524
 503.5635189  497.15477459 488.87681318 487.00759609 480.86588279
 473.92307645 470.18464227 462.97480492 459.77043276 456.03199858
 449.35622325 441.87935489 429.8629593  424.25530803 419.7157808
 409.3015713  402.62579597 395.14892761 390.34236937 383.66659404
 374.5875396  368.4458263  366.3095782  359.90083388 357.49755477
 352.15693451 349.48662438 344.14600412 339.87350791 336.66913575
 328.65820536 325.72086422 324.91977118 315.84071673 312.10228255
 309.96603445 306.22760026 300.61994899 298.21666987 292.34198759
 287.268398

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Warning in <Fit>: Abnormal termination of minimization.
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2


In [226]:
if parameters[0] - 1 < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if parameters[1] < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if parameters[2] - 1 < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const è compatibile con 1
---
mu è compatibile con 0
---
sigma NON è compatibile con 1


In [227]:
def line(x, par):
    x = x[0]
    m = par[1]
    q = par[2]
    return m * x + q

residuals = q - model(p, par)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Dati con errori; x; y")
gr2.SetMarkerStyle(20)
gr2.SetMarkerColor(ROOT.kBlue)
gr2.SetLineColor(ROOT.kBlack)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")


c3.Draw()
c3.SaveAs("fit_line.png")

Fit status: 2
Parametri stimati:
m = 1.0 ± 0.0
q = 0.9233116997252216 ± 0.0
Chi2 = 21124407.30256023
****************************************
         Invalid FitResult  (status = 2 )
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =  2.11244e+07
NDf                       =          122
Edm                       =   8.3353e-17
NCalls                    =           27
m                         =            1   +/-   0           
q                         =     0.923312   +/-   0           


Warning in <Fit>: Abnormal termination of minimization.
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
